In [1]:
import pandas as pd
import numpy as np
import os
import ast
from tqdm.notebook import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

from astropy.visualization import make_lupton_rgb
from astropy.io import fits

import matplotlib.pyplot as plt
plt.style.use('dark_background')

In [2]:
class RealEuclidDataset(Dataset):
    
    def __init__(self, data_path: str, split: str='train', img_size: int = 64, **kwargs):
        super().__init__()
        self.images = np.load(data_path + 'images.npy') #images
        self.data = pd.read_csv(data_path + 'data_fixed.csv')
        self.num_samples = len(self.images)
        
        self.transforms = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomApply([transforms.RandomRotation((90, 90))]),
            transforms.Resize((img_size, img_size)),
            transforms.Normalize(mean=(0.5, ), std=(0.5, )),])

        # Compute stretch bounds from the FULL dataset (or training split only)
        self._compute_stretch_params()

        
    def __len__(self):
        return self.num_samples

    def _compute_stretch_params(self):
        low  = np.percentile(self.images, 1)
        high = np.percentile(self.images, 99)
        self.low  = low
        self.high = high
        self.log_min = np.log1p(0)              # = 0, since we shift by low
        self.log_max = np.log1p(high - low)     # maximum possible after shift
    
    def __getitem__(self, idx):
        image = self.images[idx].copy() 
        
        # Transforming the astronomical images
        # 1. Clip outliers using dataset-level percentiles
        image = np.clip(image, self.low, self.high)
        # 2. Shift so minimum maps to 0, then log stretch
        image = np.log1p(image - self.low)
        # 3. Scale to [0, 1] using dataset-level log_max (log_min is 0 so can be omitted)
        image = (image - self.log_min) / (self.log_max - self.log_min)
        image = np.clip(image, 0, 1)
        # 5. To tensor
        tensor = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
        # 6. Final transf. (includes normalization [-1,1])
        tensor = self.transforms(tensor)

        data_point = self.data.iloc[idx].to_dict()
        return {'image': tensor, 'data': data_point}

In [3]:
path_data = '/Users/jimenagonzalez/research/SkAI_research/SkAI_SLmodeling/Data/Euclid_morphological_catalog/cutouts_vis/'
dataset = RealEuclidDataset(path_data)
print(dataset.images.shape)

(8887, 99, 99)


In [4]:
train_loader = torch.utils.data.DataLoader(dataset=dataset, batch_size=10, num_workers=0, shuffle=True)

In [5]:
for i, sample in enumerate(tqdm(train_loader)):
    image_batch, data_batch = sample['image'], sample['data']
    print(image_batch.shape)

  0%|          | 0/889 [00:00<?, ?it/s]

torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 64, 64])
torch.Size([10, 1, 6